# Feature Engineering : Préparation pour le Machine Learning

Ce notebook prend le fichier  et le transforme en , prêt à être ingéré par des modèles prédictifs (XGBoost, Random Forest, Régression Linéaire).

In [ ]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler, MinMaxScaler

# Configuration
pd.set_option("display.max_columns", None)

print("Libraries chargées avec succès.")
import re
from sklearn.feature_extraction.text import TfidfVectorizer


## 1. Chargement des données propres

In [ ]:
df = pd.read_csv("../data/clean_data.csv")
print(f"Dimensions: {df.shape[0]} lignes, {df.shape[1]} colonnes.")
df.head(3)

## 1.5. Extraction de Caractéristiques (NLP sur la Description)
On extrait d'abord des **métriques structurelles** : la longueur, les hashtags, les liens...

In [ ]:
# Fonctions de calcul NLP
def count_links(text):
    return len(re.findall(r'http[s]?://', str(text)))

def count_hashtags(text):
    return len(re.findall(r'#(\w+)', str(text)))

def shouting_ratio(text):
    text_str = str(text)
    if len(text_str) == 0: return 0
    return sum(1 for c in text_str if c.isupper()) / len(text_str)

df['desc_length'] = df['description'].fillna('').apply(lambda x: len(str(x)))
df['desc_num_hashtags'] = df['description'].fillna('').apply(count_hashtags)
df['desc_has_links'] = df['description'].fillna('').apply(count_links)
df['desc_shouting_ratio'] = df['description'].fillna('').apply(shouting_ratio)

print("Métriques structurelles ajoutées !")
df[['description', 'desc_length', 'desc_num_hashtags', 'desc_has_links']].head(3)

Ensuite, on utilise **TF-IDF** pour trouver les 50 mots les plus discriminants (mots-clés).

In [ ]:
# On remplace les NaN par du texte vide ou notre token
texts = df['description'].fillna('')

from sklearn.feature_extraction.text import TfidfVectorizer
tfidf = TfidfVectorizer(max_features=50, stop_words='english')
tfidf_matrix = tfidf.fit_transform(texts)
tfidf_df = pd.DataFrame(tfidf_matrix.toarray(), columns=[f"word_{w}" for w in tfidf.get_feature_names_out()])

# On ajoute ces 50 colonnes au DataFrame principal
df = pd.concat([df, tfidf_df], axis=1)

print(f"{tfidf_df.shape[1]} Mots-clés extraits via TF-IDF !")
print("Exemple de mots :", list(tfidf_df.columns)[:10])

## 2. Sélection des Variables (Feature Selection)
On retire les identifiants textuels (qui ne servent à rien pour l'algorithme) et les variables temporelles brutes.

In [ ]:
# 2. Sélection des Variables (Feature Selection)
cols_to_drop = [
    # Identifiants textuels inutiles pour l'algorithme
    "id", "title", "description", "publishedAt", 
    "channelId", "channel_id", "channel_name", 
    "date", "thumbnails", "tags", "topics", "duration", "creator_tier"
    # Constantes mathématiques vides (Zero-Variance Features)
    "avg_like_rate", "avg_duration_sec", "pct_vertical", 
    "pct_has_music", "viral_video_rate", "consistency_score"
]

# On garde df_features pour l'entraînement pur
df_features = df.drop(columns=cols_to_drop, errors="ignore")

print(f"Colonnes conservées pour l'apprentissage : {df_features.shape[1]}")
list(df_features.columns)


## 3. Encodage des Variables Catégoriques
L'algorithme ne comprend que les chiffres. Si nous avons des catégories (comme le ), il faut les transformer en 0, 1, 2, etc.

In [ ]:
# # Vérifions s'il y a des colonnes non numériques
# categorical_cols = df_features.select_dtypes(include=["object"]).columns
# print("Colonnes nécessitant un encodage :", list(categorical_cols))

# # Si creator_tier est présent, on peut faire un One-Hot Encoding ou un Label Encoding
# if "creator_tier" in categorical_cols:
#     # Get Dummies convertit "Bronze", "Gold" en colonnes tier_Bronze (0/1), tier_Gold (0/1)
#     df_features = pd.get_dummies(df_features, columns=["creator_tier"], drop_first=True)
#     print("creator_tier encodé avec succès.")

# df_features.head()

## 4. Nettoyage des fuites de données (Data Leakage)
**ATTENTION CRITIQUE** : La variable cible  a été calculée mathématiquement à partir de , , et .
Si on laisse  et  dans les variables d'entraînement, l'algorithme va "tricher" à 100% en devinant la formule ! Seule compte la capacité à prédire la viralité A PARTIR des attributs de la vidéo (titre long, émoticônes, âge de la chaîne...), et non à partir de ses performances futures.

In [ ]:
leakage_cols = ["views", "likes", "comments", "views_per_subscriber"]
df_ml = df_features.drop(columns=[col for col in leakage_cols if col in df_features.columns])

print(f"Attributs prédictifs finaux ({df_ml.shape[1]} colonnes) :")
print(list(df_ml.columns))

## 5. Corrélation avec la Target
Regardons quelles sont les variables qui influencent le plus le score de viralité !

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import seaborn as sns
import numpy as np

plt.figure(figsize=(14, 10))

# 1. Calcul de la matrice de corrélation complète
corr_matrix = df_ml.select_dtypes(include=['number']).corr()
# 2. Sélection des 20 variables les plus corrélées (en absolu) avec le virality_score
# Cela nettoie le graphique en retirant le 'bruit' (les 40 mots-clés TF-IDF qui ont 0.00 de corrélation)
top_features = corr_matrix['virality_score'].abs().sort_values(ascending=False).head(20).index
corr_matrix_top = df_ml[top_features].corr()

# 3. Masque pour cacher la moitié supérieure
mask = np.triu(np.ones_like(corr_matrix_top, dtype=bool))

# Heatmap seaborn de la matrice filtrée
ax = sns.heatmap(corr_matrix_top, mask=mask, annot=True, fmt=".2f", cmap="coolwarm", 
                 vmax=1, vmin=-1, center=0, square=True, linewidths=.5, cbar_kws={"shrink": .5})

# --- HIGHLIGHT DU SCORE DE VIRALITÉ ---
if 'virality_score' in corr_matrix_top.columns:
    target_idx = corr_matrix_top.columns.get_loc("virality_score")
    ax.add_patch(patches.Rectangle((0, target_idx), target_idx + 1, 1, fill=False, edgecolor='lime', lw=4, clip_on=False))
    ax.add_patch(patches.Rectangle((target_idx, target_idx), 1, len(corr_matrix_top) - target_idx, fill=False, edgecolor='lime', lw=4, clip_on=False))

plt.title("Top 20 des attributs les plus corrélés à la Viralité (- de bruit)", fontsize=16, fontweight="bold")
plt.tight_layout()
plt.show()


## 6. Sauvegarde des Features
Les données sont prêtes à être envoyées à Scikit-Learn ou XGBoost.

In [ ]:
df_ml.to_csv("../data/features.csv", index=False)
print("Fichier features.csv sauvegardé avec succès dans le dossier data ! ")